# Annotation listing and navigation

This notebook imports pUC19 from a GenBank file, loads its feature annotations,
and navigates to the MCS (multiple cloning site) using `widget.go_to()` and
`widget.show()`.

In [17]:
import os
import tempfile
import gen

## Import pUC19

Create a fresh in-memory repository and import the GenBank fixture.

In [18]:
# Adjust this path if running from outside the project root.
FIXTURE = os.path.abspath("../../fixtures/puc19.gb")

tmp = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmp, ".gen"))
repo.import_genbank(FIXTURE)

bg = repo.get_block_groups()[0]
print("Block group:", bg.name)

Block group: sequence-222046-


## Plot the graph

Render the sequence graph in a widget, then load the GenBank feature annotations
as a track panel.

In [24]:
widget = bg.plot(rows=24)

# The annotation group name is generated from the collection, sample, and locus
# name embedded in the GenBank file.
annotation_group = repo.query("SELECT name FROM annotation_groups")[0][0]
print("Annotation group:", annotation_group)
widget.add_annotation_track_group(annotation_group)

widget

Annotation group: GenBank default/reference/sequence-222046-


## List all annotations

`widget.list_annotations()` returns `AnnotationRecord` objects spanning all
sources (track panels and inline highlights).  Each record exposes `.name`,
`.track`, `.type`, and a full `.locus`.

In [25]:
anns = widget.list_annotations()
print(f"{len(anns)} annotations loaded")
for a in anns:
    print(f"  {a.name!r:30s}  {a.type}  {a.locus}")

21 annotations loaded
  'source'                        track  GraphLocus(348d9388[0..2686]+0 → 348d9388[0..2686]+2686, 1 blocks, strand=+)
  'pBR322ori-F'                   track  GraphLocus(348d9388[0..2686]+117 → 348d9388[0..2686]+137, 1 blocks, strand=+)
  'L4440'                         track  GraphLocus(348d9388[0..2686]+370 → 348d9388[0..2686]+388, 1 blocks, strand=+)
  'CAP binding site'              track  GraphLocus(348d9388[0..2686]+504 → 348d9388[0..2686]+526, 1 blocks, strand=+)
  'lac promoter'                  track  GraphLocus(348d9388[0..2686]+540 → 348d9388[0..2686]+571, 3 blocks, strand=+)
  'lac operator'                  track  GraphLocus(348d9388[0..2686]+578 → 348d9388[0..2686]+595, 1 blocks, strand=+)
  'M13/pUC Reverse'               track  GraphLocus(348d9388[0..2686]+583 → 348d9388[0..2686]+606, 1 blocks, strand=+)
  'M13 rev'                       track  GraphLocus(348d9388[0..2686]+602 → 348d9388[0..2686]+619, 1 blocks, strand=+)
  'M13 Reverse'            

## Navigate to the MCS

Filter for the MCS feature and navigate to it two ways:

* `widget.go_to(record)` — left-pins the annotation start at column 12, no highlight
* `widget.show(record)` — centres the camera and adds a highlight

In [26]:
mcs = next(a for a in anns if a.name == "MCS")
print("MCS locus:", mcs.locus)
print("MCS start:", mcs.locus.start())

MCS locus: GraphLocus(348d9388[0..2686]+631 → 348d9388[0..2686]+688, 1 blocks, strand=+)
MCS start: GraphPos(348d9388[0..2686] +631)


In [27]:
# Left-pin the MCS start at column 12 from the left edge.
widget.go_to(mcs)
widget

In [23]:
# Centre on the MCS and add a highlight.
widget.show(mcs)
widget

## Filter and batch-navigate

You can filter the list and pass any record directly to `go_to` or `show`.

In [39]:
promoters = [a for a in anns if "promoter" in a.name.lower()]
print("Promoters:", [a.name for a in promoters])

if promoters:
    widget.show(promoters[0])
    widget

widget._controller.get_inline_annotation_names()

Promoters: ['lac promoter', 'AmpR promoter']


'["","","",""]'

## Navigate to inline annotations by name

Search for a sequence, add the matches as named inline annotations, then navigate
to one by passing the name string to `widget.go_to()`.

In [ ]:
# Search for the lac operator sequence in pUC19.
results = repo.search("AATTGTGAGCGGATAACAATT")
matches = [locus for _, loci in results for locus in loci]
print(f"{len(matches)} match(es) found")

# Wrap each match in a named Annotation and add it inline.
for locus in matches:
    ann = gen.Annotation(locus, "lac-op-hit")
    widget.add_annotation(ann)

# list_annotations() returns AnnotationRecord objects spanning tracks + inline.
records = widget.list_annotations()
print(f"{len(records)} total records (track + inline)")

# Filter and navigate — pass the record directly to go_to().
hit = next(r for r in records if r.name == "lac-op-hit")
print(f"Navigating to: {hit!r}")
widget.go_to(hit)
widget